In [2]:
import pandas as pd

# Ruta gestionada por DVC
df = pd.read_csv(r"C:\Users\Elenita\Desktop\proyectofinal4\dp261-g3\data\raw\04-default_credit.csv")

df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
#BALANCEO DE CLASES

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.combine import SMOTEENN, SMOTETomek
import os

# --- 2. Variables predictoras y target ---
X = df.drop("default payment next month", axis=1)
y = df["default payment next month"]

# --- 3. Verificar ratio de clases ---
print("Distribución original:", y.value_counts(normalize=True))

# --- 4. Train/Test split con stratify=y ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# --- 5. Definir técnicas de balanceo ---
samplers = {
    "SMOTE": SMOTE(random_state=42),
    "RandomOverSampler": RandomOverSampler(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
    "NearMiss": NearMiss(),
    "SMOTEENN": SMOTEENN(random_state=42),
    "SMOTETomek": SMOTETomek(random_state=42)
}

# --- 6. Modelo baseline ---
model = LogisticRegression(max_iter=1000)

# --- 7. Loop de balanceo + evaluación ---
results = []

for name, sampler in samplers.items():
    # Balancear SOLO el train
    X_res, y_res = sampler.fit_resample(X_train, y_train)

    # Entrenar modelo
    model.fit(X_res, y_res)

    # Evaluar en test desbalanceado
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]

    report = classification_report(y_test, y_pred, output_dict=True)
    auc = roc_auc_score(y_test, y_proba)
    auc_pr = average_precision_score(y_test, y_proba)

    results.append({
        "Balanceo": name,
        "Recall clase 1": report["1"]["recall"],
        "F1 clase 1": report["1"]["f1-score"],
        "ROC-AUC": auc,
        "AUC-PR": auc_pr
    })

# --- 8. Tabla comparativa ---
results_df = pd.DataFrame(results)
print(results_df)

# --- 9. Guardar datasets balanceados directamente en CSV -

for name, sampler in samplers.items():
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    balanced_df = pd.DataFrame(X_res, columns=X.columns)
    balanced_df["default payment next month"] = y_res
    balanced_df.to_csv(f"credit_balanced_{name}.csv", index=False)

Distribución original: default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64


C:\Users\Elenita\miniforge3\envs\proyecto1\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Elenita\miniforge3\envs\proyecto1\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/m

             Balanceo  Recall clase 1  F1 clase 1   ROC-AUC    AUC-PR
0               SMOTE        0.604371    0.436344  0.689313  0.467529
1   RandomOverSampler        0.611153    0.462635  0.708132  0.483223
2              ADASYN        0.596835    0.458466  0.693479  0.460903
3  RandomUnderSampler        0.636021    0.439927  0.694750  0.462157
4            NearMiss        0.680482    0.324470  0.448234  0.189145
5            SMOTEENN        0.777694    0.407101  0.692005  0.452260
6          SMOTETomek        0.568199    0.460599  0.695245  0.470988


In [4]:
# Recall clase 1 = qué tan bien detecta la clase minoritaria (los clientes que sí incumplen).
# F1 clase 1 = equilibrio entre precisión y recall para la clase minoritaria.
# ROC-AUC = capacidad general del modelo para distinguir entre clases.
# AUC-PR = rendimiento específico en la clase positiva (más sensible en datasets desbalanceados).




In [ ]:
# SMOTEENN: recall más alto (0.77), pero F1 relativamente bajo (0.40). Esto indica que detecta muchos incumplimientos, pero con bastantes falsos positivos.
# RandomOverSampler / RandomUnderSampler: métricas más equilibradas, con F1 alrededor de 0.46 y AUC-PR cercano a 0.49.
# NearMiss: recall alto (0.68), pero F1 y AUC-PR muy bajos → el modelo detecta muchos incumplimientos, pero con muy poca precisión.
# SMOTETomek / ADASYN / SMOTE: resultados intermedios, con recall y F1 aceptables, y ROC-AUC alrededor de 0.69–0.70.
# El objetivo es detectar la mayor cantidad de incumplimientos las tecnicas de blanceo mas adecuadas serian, SMOTEENN y NearMiss destacan.


